In [1]:
import cv2 as cv2
import os
import matplotlib.pyplot as plt
import numpy as np

In [2]:
video_path = "focus_video.mov"

In [3]:
def calculo_fm_total(frame:np.ndarray):
    #Convertimos a greyscale el frame
    frame_gris = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    # Aplico la transformada
    img_fft = np.fft.fft2(frame_gris)
    # Centro
    img_fft_shifted = np.fft.fftshift(img_fft)
    # Calculo AF
    AF = np.abs(img_fft_shifted)
    # Calculo M
    M = np.max(AF)
    # Calculo Th
    Th= np.sum(AF > (M/1000))
    # Calculo FM
    FM = Th / frame_gris.size
    return FM

In [4]:
def calculo_fm_roi(frame:np.ndarray, roi_percentage:float):

    roi_size = roi_percentage/100

    #Convertimos a greyscale el frame
    frame_gris = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    

    # Center of rectangle
    center_x, center_y = np.array(frame_gris.shape) // 2

    # Total width and height of rectangle
    rect_width, rect_height = (np.array(list(frame_gris.shape))*roi_size).astype(np.int64)

    # Calculate top-left and bottom-right points
    x_min = center_x - rect_width // 2
    x_max = center_x + rect_width // 2

    y_min = center_y - rect_height // 2
    y_max =  center_y + rect_height // 2

    # Extraer el rectángulo centrado
    roi = frame_gris[y_min:y_max, x_min:x_max]

    # Aplico la transformada
    img_fft = np.fft.fft2(roi)
    # Centro
    img_fft_shifted = np.fft.fftshift(img_fft)
    # Calculo AF
    AF = np.abs(img_fft_shifted)
    # Calculo M
    M = np.max(AF)
    # Calculo Th
    Th= np.sum(AF > (M/1000))
    # Calculo FM
    FM = Th / roi.size
    return FM

In [ ]:
# Metodo de absolute central moment
def calculo_acmo(frame:np.ndarray):
    #Convertimos a greyscale el frame
    frame_gris = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # Calculamos histograma
    hist = cv2.calcHist([frame_gris], [0], None, [256], [0, 256])
    hist = hist.astype(np.int64).flatten()  # Convertimos a array 1d
    total_pixels = frame_gris.size # Tamaño de imagen
    # Compute mean intensity
    intensity_values = np.arange(256)
    mean_intensity = np.sum(intensity_values * hist) / total_pixels

    probabilities = hist / total_pixels

    distance_vector = np.abs(np.arange(256)-mean_intensity)

    ACMo = np.dot(distance_vector, probabilities)

    return ACMo

In [14]:
def obtencion_resultados(video_path:str,output_path:str, fm_function):
    captura_video = cv2.VideoCapture(video_path)

    fps = int(captura_video.get(cv2.CAP_PROP_FPS))
    width = int(captura_video.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(captura_video.get(cv2.CAP_PROP_FRAME_HEIGHT))
    frame_count = int(captura_video.get(cv2.CAP_PROP_FRAME_COUNT))

    output_size = (width * 2, height)
    out = cv2.VideoWriter(output_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, output_size)

    fm_list = []

    for i in range(frame_count):
        ret, frame = captura_video.read()
        if not ret:
            break

        fm = fm_function(frame)
        fm_list.append(fm)

        # Crear figura de Matplotlib directamente aquí
        plt.figure(figsize=(5, 3))
        plt.plot(fm_list, color='blue')
        plt.xlim(0, frame_count)
        plt.ylim(np.min(fm_list), np.max(fm_list)*1.1)
        plt.title("Índice")
        plt.tight_layout()

        # Guardar figura como imagen temporal
        plt.savefig("plot_temp.png")
        plt.close()

        # Leer la imagen del plot y redimensionar
        plot_img = cv2.imread("plot_temp.png")
        plot_img = cv2.resize(plot_img, (width, height))

        # Unir lado a lado
        combined = np.hstack((frame, plot_img))
        out.write(combined)

    print(f"Maximo indice detectado en frame: {np.argmax(fm_list)}")

    captura_video.release()
    out.release()
    cv2.destroyAllWindows()

    os.remove("plot_temp.png")

In [15]:
obtencion_resultados(video_path=video_path, output_path="fm_total.mp4", fm_function=calculo_fm_total)

Maximo indice detectado en frame: 109


In [16]:
roi_percentage = 10
obtencion_resultados(video_path=video_path, output_path="fm_roi.mp4",
                     fm_function=lambda frame: calculo_fm_roi(frame, roi_percentage))

Maximo indice detectado en frame: 91


In [17]:
obtencion_resultados(video_path=video_path, output_path="fm_acmo.mp4", fm_function=calculo_acmo)


Maximo indice detectado en frame: 117
